# Module 3 — NGS Pipeline with Claude Code (Alignment)

**Sessions 6–7 | SBML Lab Intern Training | KAIST**

---

This module covers the full NGS alignment pipeline — from raw SRA data to a sorted, indexed BAM and a GFF file ready for MetaScope. You'll use Claude Code to generate the pipeline commands, but you must understand what each step does before running it.

**Lab context:** Reference genome is *E. coli* K-12 MG1655. Aligner is `bowtie2`. Post-alignment processing uses `samtools`, followed by `makegff.py` to produce a GFF file for MetaScope visualization.

**Learning objectives:**
- Understand how `bowtie2` aligns short reads to a reference genome
- Understand why BAM files must be sorted and indexed before downstream use
- Practice using Claude Code's **plan mode** for multi-step pipelines
- Annotate and verify every flag in a generated command before running it

## 1. How `bowtie2` Works

`bowtie2` aligns short reads to a reference genome using three key mechanisms. First, it builds an **FM-index** from the reference genome — a compressed, full-text index based on the Burrows-Wheeler Transform that enables fast substring lookup without scanning the entire genome for every read. Second, it uses a **seed-and-extend** strategy: short k-mer seeds from the read are looked up in the FM-index to locate candidate alignment positions, then the full alignment is extended from those positions using dynamic programming. Third, unlike its predecessor `bowtie1`, `bowtie2` supports **gapped alignment** — it can handle insertions and deletions in the read relative to the reference, which is essential for real sequencing data where small indels are common.

### Key flags to know

| Flag | Meaning |
|---|---|
| `-x` | Path to the bowtie2 index (prefix, not a file extension) |
| `-1` / `-2` | Paired-end read files (R1 and R2) |
| `-U` | Unpaired (single-end) read file |
| `-S` | Output SAM file path |
| `--no-unal` | Suppress unaligned reads from the output |
| `-p` | Number of threads |
| `--very-sensitive` | Preset for more sensitive (slower) alignment |

### Exercise 1 — Understand bowtie2 Before Running It

> **Before running anything:** use Claude Code to understand how `bowtie2` finds alignments and what the key flags control. Write a 3-sentence summary in your own words in the cell below.
>
> Suggested prompt: *"Explain how bowtie2 aligns short reads to a reference. Focus on the FM-index and seed-and-extend strategy. What does the `-x` flag point to and how is the index built?"*

*(Write your 3-sentence summary here.)*

## 2. `samtools` Internals

### SAM vs BAM

**SAM** (Sequence Alignment/Map) is a plain-text tab-delimited format. It is human-readable but large and slow to process. **BAM** is the binary-compressed equivalent — typically 3–5× smaller and orders of magnitude faster for random access, which is why all downstream tools expect BAM.

### Why sort before index?

A BAM index (`.bai`) stores **byte offsets** for fixed-size genomic bins (16 kb by default). For the index to be useful, all reads in a bin must be physically contiguous in the BAM file — i.e., reads must be **sorted by genomic coordinate**. If the BAM is unsorted (reads appear in the order they were sequenced), the index cannot map a genomic region to a unique byte range. Tools like `samtools mpileup`, `IGV`, and coverage calculations all require a coordinate-sorted + indexed BAM for this reason.

### What `.bai` stores

The `.bai` index file stores **byte offsets and virtual file offsets** for each reference sequence, divided into 16 kb bins. It also maintains a linear index at 16 kb resolution so tools can quickly seek to the first read overlapping a query region without scanning the full file.

### Exercise 2 — Sort Before Index

> **Without asking Claude Code first:** write in the cell below why you must sort a BAM file before indexing it. Use your own reasoning.
>
> Then verify your explanation with Claude Code. Prompt suggestion: *"Why does samtools index require a coordinate-sorted BAM? What happens if you try to index an unsorted BAM?"*
>
> Update your answer if needed and note what you got right or wrong.

*(Write your reasoning here before consulting Claude Code.)*

## 3. Plan Mode — Reviewing Pipelines Before Running

Claude Code's **plan mode** lets you see every step Claude intends to take before a single command is executed.

**How to activate:** Press **Shift+Tab** before sending your prompt. The interface switches to plan mode. Claude will show you the full sequence of actions — file reads, commands, edits — and wait for your approval.

**Why this matters for pipelines:**  
A multi-step NGS pipeline touches raw data, builds index files, and writes intermediate outputs. A mistake at step 2 (wrong index path) silently produces a valid-looking but incorrect BAM. Reviewing the plan lets you:
- Catch wrong paths or flag values before they cause silent errors
- Understand every command before it runs
- Modify individual steps (e.g., add `--no-unal`, increase `-p` threads) without re-generating the whole plan

**Rule for this module:** Any time you ask Claude Code to generate a multi-step pipeline, you must use plan mode and review it before approving.

### Exercise 3 — Plan Mode Pipeline Review

> Use plan mode to generate the full pipeline:
> ```
> FASTQ → bowtie2 align → samtools sort → samtools index → makegff.py
> ```
> Suggested prompt (use Shift+Tab first):
> *"Generate a shell pipeline for E. coli K-12 MG1655: build a bowtie2 index from the reference FASTA, align paired-end FASTQs, convert to BAM, sort, index, then run makegff.py on the BAM. Use sample paths."*
>
> Review the proposed steps. **Identify at least one flag or parameter you would change or verify before running.** Write what you changed and why in the cell below.

*(Write the flag/parameter you identified, your change, and your reasoning here.)*

## 4. SRA Download with `fastq-dump`

`fastq-dump` (part of the SRA Toolkit) downloads sequencing data from NCBI's Sequence Read Archive by accession number (e.g., `SRR12345678`).

The most important flag for paired-end data is `--split-files`. Without it, `fastq-dump` interleaves both reads into a single FASTQ file — a format most aligners do not accept as paired-end input. With `--split-files`, it writes `_1.fastq` and `_2.fastq` separately, which is what `bowtie2 -1` and `-2` expect.

Other useful flags: `--gzip` (compress output on the fly), `--outdir` (specify output directory), `--skip-technical` (drop technical reads like barcodes).

In [ ]:
%%bash
# Exercise 4 — Generate the fastq-dump command for a given SRR accession using Claude Code.
# Paste the command here and annotate each flag with a comment.
#
# Prompt suggestion: "Write a fastq-dump command to download SRR12345678 as paired-end FASTQs,
# compressed with gzip, into a directory called data/raw/. Explain each flag."

# fastq-dump [flags] <SRR_accession>


## 5. SAM FLAG Values

The SAM **FLAG** field is a 12-bit integer where each bit encodes a binary alignment property. The value you see in column 2 of a SAM record is the sum of all set bits.

| Value | Bit | Meaning |
|---|---|---|
| 1 | 0x1 | Read is paired |
| 2 | 0x2 | Read mapped in proper pair |
| 4 | 0x4 | Read unmapped |
| 8 | 0x8 | Mate unmapped |
| 16 | 0x10 | Read on reverse strand |
| 32 | 0x20 | Mate on reverse strand |
| 64 | 0x40 | First in pair (R1) |
| 128 | 0x80 | Second in pair (R2) |
| 83 | — | Paired, proper pair, reverse strand, first in pair (common R1 value) |
| 163 | — | Paired, proper pair, mate on reverse strand, second in pair (common R2 value) |

You can decode any FLAG at [broadinstitute.github.io/picard/explain-flags.html](https://broadinstitute.github.io/picard/explain-flags.html) or by asking Claude Code: *"Decode SAM FLAG 2064."*

### Exercise 5 — Decode an Unknown FLAG

> Look up what a specific FLAG value means using Claude Code. Choose a value you have **not** seen in the table above (e.g., 2048, 256, 1024).
>
> Write:
> 1. The FLAG value you chose
> 2. What it means (decoded bits)
> 3. How you would verify that a real SAM record carries this flag (e.g., with `samtools view -f` or `samtools flags`)

*(Write your FLAG value, its meaning, and verification method here.)*

## 6. Running the Full Pipeline on Sample Data

Sample data is in `data/sample/`. The directory contains:
- `ecoli_k12.fasta` — *E. coli* K-12 MG1655 reference genome
- `sample_R1.fastq.gz` and `sample_R2.fastq.gz` — paired-end reads

**Use plan mode before running.** Generate the full pipeline with Shift+Tab, review every step, verify all paths and flags, then approve.

Fill in each step in the code cell below after generating and reviewing it with Claude Code.

In [ ]:
%%bash
# Full NGS pipeline — fill in the commands after generating them with Claude Code in plan mode.
# Do NOT run this cell until you have reviewed the plan and annotated each flag.

# Step 1: Build bowtie2 index from reference FASTA


# Step 2: Align paired-end reads with bowtie2


# Step 3: Convert SAM to BAM (samtools view)


# Step 4: Sort BAM by coordinate (samtools sort)


# Step 5: Index sorted BAM (samtools index)


# Step 6: Run makegff.py to generate GFF for MetaScope


## End of Session

Before closing:

1. Make sure your pipeline cell runs end-to-end without errors.
2. Verify that the output BAM is coordinate-sorted: `samtools view -H output.bam | grep SO`
3. Verify the index exists: `ls -lh output.bam.bai`
4. Run `/log` in Claude Code to save a session log.

---

**Run `/log` before closing.**